# FedSecureAI — Full LI-Medium Federated Learning GPU Runner

This notebook executes the full **12 federated learning configurations** on the complete **31.25-million-row IBM AML LI-Medium dataset** (~18.75M train, ~6.25M val, ~6.25M test) on Google Colab with GPU acceleration.

### Key Capabilities & Architectural Safeguards:
1. **Memory-Mapped Sequence Architecture (`np.memmap`)**:
   - Zero pandas DataFrames in memory during training.
   - Resident host RAM is bounded to **< 1.0 GB** throughout the entire 31.25M-row training run, completely preventing Linux OOM killer (`SIGKILL`).
2. **Zero-Copy Federated Client Partitioning**:
   - Clients reference shared memory-mapped feature arrays on disk using compact integer row indices.
   - Preserves strict Non-IID account typology isolation (zero sequences cross client boundaries) and IID stratified class balance.
3. **Persistent Google Drive Checkpointing & Atomic Resumption**:
   - Dedicated `resume_manifest.json` under `artifacts/full_li_medium_experiment/`.
   - Per-round checkpoints saved at `round_{r}.pt` and final `global_best.pt`.
   - Safely survives Colab disconnections or timeouts; re-running automatically resumes from the exact saved checkpoint.
4. **Full 12-Experiment Matrix**:
   - **Architectures**: GRU, LSTM, BiLSTM
   - **Algorithms**: FedAvg, FedProx ($\mu=0.0001$)
   - **Distributions**: IID, Non-IID (3 heterogeneous clients: Retail Giant, Offshore/Crypto, Regional Credit Union)
   - **Communication**: 5 global rounds, 1 local epoch per round, batch size 4096 on GPU.


## 1. Environment & GPU Hardware Verification

In [ ]:
!nvidia-smi
import psutil
import shutil
import torch

print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU VRAM        : {vram:.2f} GB")
ram = psutil.virtual_memory().total / (1024**3)
disk = shutil.disk_usage("/").free / (1024**3)
print(f"Host System RAM : {ram:.2f} GB")
print(f"Available Disk  : {disk:.2f} GB")
print("=" * 60)


## 2. Mount Google Drive for Persistent Storage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Extract Colab-Ready Code Package

In [ ]:
import os
from pathlib import Path

# Working directories
DRIVE_ROOT = Path("/content/drive/MyDrive/FedSecureAI")
LOCAL_ROOT = Path("/content/FedSecureAI")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(str(LOCAL_ROOT))

# Locate ZIP package
zip_candidates = [
    DRIVE_ROOT / "FedSecureAI_FULL_LI_MEDIUM_COLAB_READY_v1.zip",
    Path("/content/FedSecureAI_FULL_LI_MEDIUM_COLAB_READY_v1.zip"),
    DRIVE_ROOT / "FedSecureAI_colab.zip",
    Path("/content/FedSecureAI_colab.zip"),
]

zip_path = next((p for p in zip_candidates if p.exists()), None)
if zip_path:
    print(f"Extracting {zip_path} into {LOCAL_ROOT}...")
    !unzip -q -o "{zip_path}" -d "{LOCAL_ROOT}"
    print("Codebase extraction complete.")
else:
    print("WARNING: Package ZIP not found! Please upload FedSecureAI_FULL_LI_MEDIUM_COLAB_READY_v1.zip to /content/ or Google Drive.")


## 4. Install Dependencies

In [ ]:
!pip install -q -r requirements_colab.txt


## 5. Configure Full LI-Medium Dataset & Persistent Output Directory

In [ ]:
from pathlib import Path

# Locate LI-Medium dataset
data_candidates = [
    DRIVE_ROOT / "data" / "processed" / "ibm_aml" / "LI-Medium",
    LOCAL_ROOT / "data" / "processed" / "ibm_aml" / "LI-Medium",
    Path("/content/data/processed/ibm_aml/LI-Medium"),
]

DATA_PATH = next((p for p in data_candidates if (p / "train" / "transactions.csv").exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not locate the full LI-Medium dataset with train/transactions.csv! "
        "Please ensure your full LI-Medium dataset folder is present in Google Drive at "
        "/content/drive/MyDrive/FedSecureAI/data/processed/ibm_aml/LI-Medium or /content/data/."
    )

print(f"Active Dataset Root : {DATA_PATH}")

# Set persistent output directory in Google Drive
OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "full_li_medium_experiment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Persistent Output Dir : {OUTPUT_DIR}")


## 6. Run Unit & Integration Tests
Validates the memory-mapped sequence provider, zero-copy client partitioning, strict full-dataset enforcement, and FedAvg/FedProx aggregation before running full training.

In [ ]:
!pytest tests/test_full_li_medium_pipeline.py -v


## 7. Preflight Verification (Strict Full-Dataset Inspection)
Verifies that all 18.75M train rows, 6.25M val rows, 6.25M test rows, and 57 features are present without truncation.

In [ ]:
!python backend/scripts/run_full_li_medium_training.py \
    --data-root "{DATA_PATH}" \
    --output-dir "{OUTPUT_DIR}" \
    --device auto \
    --verify-only


## 8. Benchmark Single Configuration on GPU (Config 1: IID GRU FedAvg)
Runs 5 global communication rounds for Configuration 1 to verify GPU cuDNN throughput and memory stability (~3-4 minutes).

In [ ]:
!python backend/scripts/run_full_li_medium_training.py \
    --data-root "{DATA_PATH}" \
    --output-dir "{OUTPUT_DIR}" \
    --device auto \
    --batch-size 4096 \
    --rounds 5 \
    --config-id 1


## 9. Execute Full 12-Experiment Suite (Atomic Resumption Enabled)
Sequentially trains all 12 configurations across GRU, LSTM, BiLSTM $\times$ FedAvg, FedProx ($\mu=0.0001$) $\times$ IID, Non-IID.
- Already completed configurations are automatically skipped.
- In case of disconnection, re-running this cell resumes from the last completed configuration or round.

In [ ]:
!python backend/scripts/run_full_li_medium_training.py \
    --data-root "{DATA_PATH}" \
    --output-dir "{OUTPUT_DIR}" \
    --device auto \
    --batch-size 4096 \
    --rounds 5


## 10. Inspect Final Metrics & Status Report

In [ ]:
import pandas as pd
status_csv = OUTPUT_DIR / "results" / "experiment_status.csv"
if status_csv.exists():
    df = pd.read_csv(status_csv)
    display(df)
else:
    print("Status CSV not found yet.")


## 11. Package All Experiment Outputs into a Downloadable ZIP

In [ ]:
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_zip = DRIVE_ROOT / f"FedSecureAI_FULL_LI_MEDIUM_RESULTS_{timestamp}.zip"
print(f"Packaging {OUTPUT_DIR} to {output_zip}...")
!zip -r -q "{output_zip}" "{OUTPUT_DIR}"
print(f"ZIP package created successfully at: {output_zip}")
